In [2]:
# 导入库

import pandas as pd
import pickle
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

In [3]:
# 导入数据

test_df = pd.read_csv('./test.csv', encoding='UTF-8')
test_df.head()

,user_id,video_id,name_en,description_en
0,191776,13,Alive!,"In this compelling documentary, five HIV-posit..."
1,144905,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...
2,149531,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...
3,69657,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...
4,201881,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...


In [7]:
# 分词 - 词形还原 - 去停用词

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return None
    
    
def preprocessing_text(x, stop_words):
    arr = word_tokenize(x)
    tag_arr = pos_tag(arr)
    wnl = WordNetLemmatizer()
    lemm_arr = []
    for tag in tag_arr:
        wordnet_pos = get_wordnet_pos(tag[1]) or wordnet.NOUN
        lemm_arr.append(wnl.lemmatize(tag[0], wordnet_pos))
    return ' '.join([i for i in lemm_arr if i not in stop_words])


with open('./nlp/stop_words_en.txt', 'r', encoding='UTF-8') as f:
    stop_words = f.read().split('\n')
    
test_df['text'] = test_df['name_en'] + test_df['description_en']
test_df['text'] = test_df['text'].apply(lambda x: preprocessing_text(x, stop_words))
test_df.head()

,user_id,video_id,name_en,description_en,text
0,191776,13,Alive!,"In this compelling documentary, five HIV-posit...",Alive In compelling documentary five HIV-posit...
1,144905,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,Appropriate BehaviorShirin struggle become ide...
2,149531,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,Appropriate BehaviorShirin struggle become ide...
3,69657,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,Appropriate BehaviorShirin struggle become ide...
4,201881,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,Appropriate BehaviorShirin struggle become ide...


In [8]:
# 非结构数据转换为结构数据

with open('./vec_method.pickle', 'rb') as f:
    vec_method = pickle.load(f)
    
word_mtx = vec_method.transform(test_df['text'])
word_mtx.shape

(94089, 8510)

In [9]:
# 特征选择

with open('./de_method.pickle', 'rb') as f:
    de_method = pickle.load(f)
    
word_mtx_de = de_method.transform(word_mtx)
word_mtx_de.shape

(94089, 6300)

In [12]:
# 模型导入

with open('./clf.pickle', 'rb') as f:
    clf = pickle.load(f)
    
Y_pred = clf.predict(word_mtx_de)

In [15]:
# 保存结果

temp = []
result = []

for i in range(len(test_df['user_id'])):
    temp.append(test_df['user_id'][i])
    temp.append(test_df['video_id'][i])
    if Y_pred[i] > 5:
        Y_pred[i] = 5
    temp.append('%.0f'%Y_pred[i])
    result.append(temp)
    temp = []
    
result_csv = pd.DataFrame(data=result)
result_csv.to_csv('./徐茂凌_result.csv', header=['user_id', 'video_id', 'predicted_rank'], index=False, encoding='utf_8_sig')
result_csv.head()

,0,1,2
0,191776,13,1
1,144905,14,2
2,149531,14,2
3,69657,14,2
4,201881,14,2
